<a href="https://colab.research.google.com/github/tribhubaneshward/PennyCMP/blob/main/First%20pennylane%20code%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pennylane as qml
from pennylane import numpy as np

symbols_A = ["H", "Cl"]
coordinates_A = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.275])

H, qubits = qml.qchem.molecular_hamiltonian(
    symbols_A,
    coordinates_A,
)

electrons = 18
hf_state = qml.qchem.hf_state(electrons, qubits)

# FIX: Use the updated single unified excitations function
singles, doubles = qml.qchem.excitations(electrons, qubits)

dev = qml.device("default.qubit", wires=qubits)

@qml.qnode(dev)
def circuit(weights):
    qml.BasisState(hf_state, wires=range(qubits))
    qml.AllSinglesDoubles(
        weights,
        wires=range(qubits),
        hf_state=hf_state,
        singles=singles,
        doubles=doubles
    )
    return qml.expval(H)

# Correctly initialize weights with the total number of excitations
weights = np.zeros(len(singles) + len(doubles), requires_grad=True)

opt = qml.GradientDescentOptimizer(stepsize=0.4)

for i in range(100):
    weights, energy = opt.step_and_cost(circuit, weights)

    if i % 10 == 0:
        print(f"Step {i}: Energy = {energy:.8f} Ha")

ValueError: The requested basis set data is not available for Cl. Please consider using `load_data=True` to download the basis set from the external library basis-set-exchange that can be installed with: pip install basis-set-exchange.

In [ ]:





from pyscf import gto, scf

# 1. Define the molecule (Geometry and Basis Set)
mol = gto.M(
    atom='H 0 0 0; H 0 0 0.74',  # Atoms and their 3D coordinates
    basis='sto-3g'               # The orbital approximation basis
)

# 2. Run a Mean-Field Hartree-Fock calculation
mf = scf.HF(mol)
energy = mf.kernel()

print(f"Classical HF Energy: {energy} Hartree")




converged SCF energy = -1.11675930739643
Classical HF Energy: -1.1167593073964255 Hartree


In [ ]:
pip install pyscf openfermion openfermionpyscf pennylane basis-set-exchange

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.3 MB/s eta 0:00:00


In [ ]:
import pennylane as qml
from openfermion.ops import FermionOperator
from openfermion.transforms import jordan_wigner

# 1. Define active space dimensions (2 electrons across 2 spatial orbitals = 4 qubits)
active_electrons = 2
active_orbitals = 2
qubits = active_orbitals * 2

# 2. Extract core integrals and slice to fit the active space size
h1 = mf.get_hcore()[:active_orbitals, :active_orbitals]
h2 = mf._eri[:active_orbitals, :active_orbitals, :active_orbitals, :active_orbitals]

# 3. Construct Second-Quantized Hamiltonian
fermion_hamiltonian = FermionOperator((), hf_energy)

for p in range(active_orbitals):
    for q in range(active_orbitals):
        fermion_hamiltonian += FermionOperator(f"{2*p}^ {2*q}", h1[p, q])
        fermion_hamiltonian += FermionOperator(f"{2*p+1}^ {2*q+1}", h1[p, q])

for p in range(active_orbitals):
    for q in range(active_orbitals):
        for r in range(active_orbitals):
            for s in range(active_orbitals):
                val = 0.5 * h2[p, q, r, s]
                fermion_hamiltonian += FermionOperator(f"{2*p}^ {2*q}^ {2*r} {2*s}", val)

# 4. Convert directly to PennyLane layout
qubit_hamiltonian = jordan_wigner(fermion_hamiltonian)
H = qml.from_openfermion(qubit_hamiltonian)

print(f"\nSuccessfully compiled crystal properties to PennyLane!")
print(f"Number of qubits: {qubits}")

# 5. Run the Quantum VQE Evaluation
dev = qml.device("default.qubit", wires=qubits)
hf_state = qml.qchem.hf_state(active_electrons, qubits)
singles, doubles = qml.qchem.excitations(active_electrons, qubits)

@qml.qnode(dev)
def circuit(weights):
    qml.BasisState(hf_state, wires=range(qubits))
    qml.AllSinglesDoubles(weights, wires=range(qubits), hf_state=hf_state, singles=singles, doubles=doubles)
    return qml.expval(H)

weights = np.zeros(len(singles) + len(doubles), requires_grad=True)
print(f"Initial Quantum VQE Energy: {circuit(weights):.6f} Ha")

IndexError: too many indices for array: array is 1-dimensional, but 4 were indexed